# 🎯 Lecture 01: Practice Problems — Technical Case Studies

Welcome to the **Lecture 01 Practice Problems Notebook**. This notebook contains three industrial-grade case studies investigating the systems-level foundations of modern Python in AI/ML:

- **Case 1**: Computational Performance Analysis — Python Dynamic Overhead vs Vectorized C-Extensions
- **Case 2**: Python Memory Management, Object Overhead & Garbage Collection Mechanics
- **Case 3**: Production Virtual Environment Isolation, Dependency Resolution & Deterministic Reproducibility


In [ ]:
import sys
import os
import time
import platform
import gc
import hashlib
from collections import deque, defaultdict

print(f"Python Platform: {platform.python_implementation()} {platform.python_version()} on {platform.system()} ({platform.machine()})")
print(f"64-bit Pointer Size: {sys.maxsize > 2**32} ({sys.getsizeof(1)} bytes for small int)")


## 📌 Case Study 1: Computational Performance Analysis — Python Dynamic Overhead vs Vectorized C-Extensions

### Mathematical Formulation
Consider computing the sum of element-wise reciprocals over an array $X \in \mathbb{R}^N$ of size $N = 10^7$ with $x_i = i + 1$:

$$
S = \sum_{i=1}^{N} \frac{1}{1 + x_i^2}
$$

#### Deliverables:
1. **Dynamic Bytecode Overhead:** Analyze why CPython VM introduces large overhead (type unboxing, dynamic attribute lookup, `BINARY_OP` dispatch).
2. **Arithmetic Intensity & Roofline Model:**
   $$
   I = \frac{\text{FLOPs}}{\text{Data Transferred (Bytes)}} = \frac{3N}{24N} = \frac{1}{8} = 0.125 \text{ FLOPs/byte}
   $$
3. **Amdahl's Law:** If $p = 95\%$ of an AI pipeline is accelerated with vectorized speedup $s = 65\times$:
   $$
   S_{\text{pipeline}} = \frac{1}{(1 - p) + \frac{p}{s}}
   $$
4. **Implementation:** Benchmark a pure Python loop against NumPy vectorization over $N = 10^6$ elements.


In [ ]:
# TODO: Case Study 1 Implementation Starter
import numpy as np

N_bench = 1_000_000 # Benchmark size

# 1. Pure Python Loop
def pure_python_reciprocal_sum(n):
    total = 0.0
    for i in range(1, n + 1):
        total += 1.0 / (1.0 + i * i)
    return total

# 2. Vectorized NumPy Execution
def numpy_reciprocal_sum(n):
    x = np.arange(1, n + 1, dtype=np.float64)
    # TODO: Implement vectorized expression
    pass

# Benchmark & compare execution times and compute speedup S = t_python / t_numpy


## 📌 Case Study 2: Python Memory Management, Object Overhead & Garbage Collection Mechanics

### Mathematical Formulation
In 64-bit CPython, every integer is wrapped in a `PyLongObject`:
- `ob_refcnt`: 8 bytes
- `ob_type`: 8 bytes
- `ob_size`: 8 bytes
- `ob_digit[0]`: 4 bytes + 4 bytes padding = 28 bytes minimum.

A Python `list` stores an array of 8-byte pointer references:
$$
S_{\text{total}} = S_{\text{list\_struct}} + N \cdot 8 + N \cdot S_{\text{PyLong}}
$$

For $N = 50,000,000$ integers, calculate:
1. Total memory consumption in standard Python list vs contiguous NumPy buffer (`int32`, 4 bytes/element).
2. The Memory Amplification Factor:
$$
\text{Amplification} = \frac{M_{\text{python\_list}}}{M_{\text{numpy\_buffer}}}
$$
3. Reference counting mechanics and cyclic reference deadlock proof:
$$
A.\text{child} = B \implies \text{RefCount}(B) = 2
$$
$$
B.\text{parent} = A \implies \text{RefCount}(A) = 2
$$


In [ ]:
# TODO: Case Study 2 Implementation Starter

# 1. Memory inspection using sys.getsizeof
n_test = 42
print("Size of integer 42:", sys.getsizeof(n_test), "bytes")

# 2. Simulate Cyclic References
class Node:
    def __init__(self, name):
        self.name = name
        self.child = None

# TODO: Create cyclic link, delete external references, and invoke gc.collect()


## 📌 Case Study 3: Production Virtual Environment Isolation, Dependency Resolution & Deterministic Reproducibility

### Mathematical Formulation
A dependency graph is a directed acyclic graph (DAG) $G = (V, E)$, where $u \to v$ denotes that package $u$ depends on package $v$.

#### Deliverables:
1. **Topological Ordering:** Implement Kahn's Algorithm using in-degrees:
$$
\text{InDegree}(v) = |\{u \in V \mid (u, v) \in E\}|
$$
2. **Cycle Detection:** If resolved topological sort contains fewer nodes than $|V|$, declare circular dependency error.
3. **SemVer Range Evaluation:**
   - Tilde ($\sim 1.2.3$): $[1.2.3, 1.3.0)$
   - Caret ($\wedge 1.2.3$): $[1.2.3, 2.0.0)$
4. **Cryptographic SHA-256 Lockfile Verification:** Verify package artifact integrity against tamper threats.


In [ ]:
# TODO: Case Study 3 Implementation Starter

# Dependency graph representation: adjacency list
dependencies = {
    "app": ["torch", "transformers", "accelerate"],
    "transformers": ["torch", "tokenizers"],
    "accelerate": ["torch"],
    "tokenizers": [],
    "torch": []
}

# TODO: Implement Kahn's algorithm to determine deterministic installation order
def kahns_topological_sort(deps):
    pass
